# imports

In [1]:
import sionna.rt as rt

import matplotlib.pyplot as plt
import mitsuba as mi
import numpy as np
import pandas as pd
import tensorflow as tf

from monte_carlo import run_monte_carlo_scene, create_munich_scene

print(f"Mitsuba variant: {mi.variant()}")
print(tf.config.list_physical_devices('GPU'))


I0000 00:00:1788708850.337669   25991 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788708850.380187   25991 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788708851.719028   25991 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Mitsuba variant: cuda_ad_mono_polarized
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Params

In [2]:
import ofdm_config

# Starting point only: the sweep loop re-selects band and comb spacing per trial.
# Layouts are seeded independently of both, so a seed gives identical drones on every
# waveform.
ofdm_config.select_band("28")

tx_power_dbm = 40       # total tx power in dBm, split across served drones
tx_power_w = 10 ** (tx_power_dbm / 10) / 1000
csi_error_std = 0.00    # stddev of simulated per-entry CSI estimation error (0 = perfect CSI)

# --- SCENE axes ---
# Keep the densest corner under the array's element count: multi-user ZF inverts H H^H
# over all served drones, so past ARRAY_ROWS*ARRAY_COLS users the Gram matrix is
# rank-deficient. That is 64 users at 10/28 GHz, 16 at 3.5 GHz.
#
# Poisson means over the forward sector:
#
#            r=50    r=100   r=150   r=200
#    500      1.2      5.2    11.7    20.9
#   1000      2.5     10.4    23.4    41.8
#   1500      3.7     15.5    35.2    62.6   <- ~= the 64-element limit
range_values_m = [50, 100, 200]   # sector outer radius, meters
density_values = [400]            # drones per km^2 within the sector
trials_per_combo = 1              # random layouts per (range, density) pair
base_seed = 0                     # seeds are base_seed + a running counter

# --- WAVEFORM axes (each value costs its own ray-traced trial) ---
# comb_spacing M is the sensing/comms split: raising it hands 1/M of the subcarriers
# back to comms and divides the radar's unambiguous range by the same factor, leaving
# range resolution untouched. The grid multiplies out, so these two axes make the
# sweep 9x longer.
band_values = ["3.5", "10", "28"]
comb_spacing_values = [0, 2, 4]      # 0 = comb off; see ofdm_config.select_comb_spacing

# --- DETECTOR axis (free: scored off the finished cube, no extra simulation) ---
# Every setting is applied to the same cube from the same drop, so the physics is held
# fixed across the axis and the P_d vs. false-alarm curve is a real ROC. Each dict is
# detect_targets() kwargs.
detect_variants = [
    {"pfa": 5e-9},
    {"pfa": 5e-8},
    {"pfa": 5e-7},
]

# range-azimuth heatmap per scene; leave off for a full sweep
range_azimuth_plot = False
sanity_check = False


# create scene

In [3]:
scene = create_munich_scene()

# Walls are purely specular (Sionna's default), so the city is invisible to the
# monostatic radar: returns come from the drone bodies alone, with no ground clutter.

# BS at one corner of an open plaza, boresight on the opposite corner ~91 m away --
# a narrow street would leave a beamwidth sweep no lateral room. The sampling sector
# is centred on this boresight.
bs_position = [-56.0, -84.0, 10.0]
look_at_point = [-128.0, -140.0, 15.0]


# Sanity check (single scene, range-azimuth plot)

One scene with `range_azimuth_plot=True`, so detections vs. ground truth can be
eyeballed before committing to a real (unplotted) sweep below.

In [4]:
if(sanity_check):
  sanity_rows = run_monte_carlo_scene(
    scene, bs_position=bs_position, look_at=look_at_point,
    range_m=100, density_per_km2=1000, seed=0,
    tx_power_w=tx_power_w, csi_error_std=csi_error_std,
    scene_id="sanity_check", range_azimuth_plot=True,
    cleanup=False,  # keep the drones in `scene` for the preview below
  )
  print(f"{len(sanity_rows)} drones in sanity-check scene")
  print(pd.DataFrame(sanity_rows))

  scene.preview()



# Monte Carlo sweep

In [5]:
import gc
from pathlib import Path

output_csv = "monte_carlo_kpis.csv"
Path(output_csv).unlink(missing_ok=True)   # rows are appended per scene below

# resolved up front so total_scenes counts the trials that will actually run
waveforms = [(b, m) for b in band_values for m in comb_spacing_values
             if ofdm_config.comb_spacing_is_valid(b, m)]

scene_counter = 0
total_scenes = len(range_values_m) * len(density_values) * len(waveforms) * trials_per_combo
total_drone_rows = 0
header_written = False

for band, comb_spacing in waveforms:
    # before the scene is built: create_munich_scene() reads the band at load time
    ofdm_config.select_band(band)
    ofdm_config.select_comb_spacing(comb_spacing)

    for range_m in range_values_m:
        for density_per_km2 in density_values:
            for trial in range(trials_per_combo):
                # Rebuild rather than edit one long-lived Scene: repeated edits perturb
                # the solve, leak GPU memory, and slow each successive trial.
                del scene
                gc.collect()
                scene = create_munich_scene()

                # scene axes only, so every (band, comb) re-runs identical layouts
                seed = base_seed + (
                    (range_values_m.index(range_m) * len(density_values)
                     + density_values.index(density_per_km2)) * trials_per_combo + trial
                )
                scene_id = f"r{range_m}_d{density_per_km2:g}_t{trial}"   # layout only
                try:   # one bad scene must not end an unattended sweep
                    trial_rows = run_monte_carlo_scene(
                        scene, bs_position=bs_position, look_at=look_at_point,
                        range_m=range_m, density_per_km2=density_per_km2, seed=seed,
                        tx_power_w=tx_power_w, csi_error_std=csi_error_std,
                        scene_id=scene_id, range_azimuth_plot=range_azimuth_plot,
                        band=band, comb_spacing=comb_spacing,
                        detect_kwargs=detect_variants,   # all scored off this one drop
                    )
                except Exception as exc:
                    import traceback
                    print(f"[{scene_counter+1}/{total_scenes}] {scene_id}: FAILED "
                          f"{type(exc).__name__}: {exc}")
                    traceback.print_exc()
                    trial_rows = []

                scene_counter += 1
                total_drone_rows += len(trial_rows)
                num_drones = len(trial_rows) // max(len(detect_variants), 1)
                print(f"[{scene_counter}/{total_scenes}] {scene_id}: {num_drones} drones "
                      f"x {len(detect_variants)} detector settings = {len(trial_rows)} rows")

                # straight to disk, so only one scene's rows are in memory at a time
                if trial_rows:
                    pd.DataFrame(trial_rows).to_csv(
                        output_csv, mode="a", header=not header_written, index=False
                    )
                    header_written = True
                del trial_rows
                gc.collect()

print(f"Total drone-rows written to {output_csv}: {total_drone_rows}")


[1/27] r50_d400_t0: 1 drones x 3 detector settings = 3 rows
[2/27] r100_d400_t0: 5 drones x 3 detector settings = 15 rows
[3/27] r200_d400_t0: 14 drones x 3 detector settings = 42 rows
[4/27] r50_d400_t0: 1 drones x 3 detector settings = 3 rows
[5/27] r100_d400_t0: 5 drones x 3 detector settings = 15 rows
[6/27] r200_d400_t0: 14 drones x 3 detector settings = 42 rows
[7/27] r50_d400_t0: 1 drones x 3 detector settings = 3 rows
[8/27] r100_d400_t0: 5 drones x 3 detector settings = 15 rows
[9/27] r200_d400_t0: 14 drones x 3 detector settings = 42 rows
[10/27] r50_d400_t0: 1 drones x 3 detector settings = 3 rows
[11/27] r100_d400_t0: 5 drones x 3 detector settings = 15 rows
[12/27] r200_d400_t0: 14 drones x 3 detector settings = 42 rows
[13/27] r50_d400_t0: 1 drones x 3 detector settings = 3 rows
[14/27] r100_d400_t0: 5 drones x 3 detector settings = 15 rows
[15/27] r200_d400_t0: 14 drones x 3 detector settings = 42 rows
[16/27] r50_d400_t0: 1 drones x 3 detector settings = 3 rows
[17/27] 

# Preview results

Rows were already written to `monte_carlo_kpis.csv` incrementally, one scene at a
time, during the sweep above -- nothing is held in memory to save here. This just
reads it back for a quick look.

In [6]:
df = pd.read_csv(output_csv)
print(f"{len(df)} rows on disk")
df.head()


540 rows on disk


,scene_id,seed,band,comb_spacing,pfa,num_training_cells,num_guard_cells,range_resolution_m,max_unambiguous_range_m,range_m,...,true_azimuth_deg,true_radial_velocity_mps,snr_db,sensing_snr_db,ber,throughput_mbps,detected,position_error_m,range_error_m,azimuth_error_deg
0,r50_d400_t0,0,3.5,0,5.000000e-09,"(8, 16)","(2, 4)",1.523336,4996.540967,50,...,-178.183163,8.407428,29.340291,89.574743,0.000152,365.120594,True,1.239544,1.183915,1.816837
1,r50_d400_t0,0,3.5,0,5.000000e-08,"(8, 16)","(2, 4)",1.523336,4996.540967,50,...,-178.183163,8.407428,29.340291,89.574743,0.000152,365.120594,True,1.239544,1.183915,1.816837
2,r50_d400_t0,0,3.5,0,5.000000e-07,"(8, 16)","(2, 4)",1.523336,4996.540967,50,...,-178.183163,8.407428,29.340291,89.574743,0.000152,365.120594,True,1.239544,1.183915,1.816837
3,r100_d400_t0,1,3.5,0,5.000000e-09,"(8, 16)","(2, 4)",1.523336,4996.540967,100,...,175.937963,-0.129482,19.253459,0.342280,0.000156,365.120594,False,NaN,NaN,NaN
4,r100_d400_t0,1,3.5,0,5.000000e-09,"(8, 16)","(2, 4)",1.523336,4996.540967,100,...,-144.259530,1.340942,18.676175,3.897464,0.000152,365.120594,False,NaN,NaN,NaN
